# CITS4012 A2 — Self-attention ablation and attention figures

This notebook continues from the TweetBiDAF v2 checkpoints already saved in
`MyDrive/CITS4012_A2_v2`.

- It restores the existing v2 checkpoints and results from Google Drive.
- It does **not** retrain the three v2 main-model runs.
- It trains `S_A3_no_selfattn` for seeds 1, 2 and 3.
- It then produces the attention figures and qualitative error sample.
- The test set remains locked and is not used.

Select a GPU runtime, then run every cell from top to bottom. When prompted,
upload the assignment copy of `train.csv`.


## Step 0 — Fixed experiment settings

In [1]:
# This notebook is preconfigured for the A3 self-attention ablation.
RUN_SMOKE_TEST = False
RUN_V2_MAIN = False
RUN_ABLATIONS = True
ABLATIONS_TO_RUN = ["S_A3_no_selfattn"]

SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/CITS4012_A2_v2"


## Step 1 — Runtime and restore from Google Drive

In [2]:
import os, sys, subprocess, shutil, json, glob, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
assert IN_COLAB, "This notebook is designed for Google Colab."
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gensim", "emoji", "nltk", "matplotlib"], check=True)

import torch
assert torch.cuda.is_available(), "GPU is not enabled. Select Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0), "| torch:", torch.__version__)

WORK = Path("/content/A2_v2")
for p in (WORK / "src", WORK / "scripts", WORK / "data", WORK / "results" / "logs"):
    p.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP = Path(DRIVE_DIR)
    BACKUP.mkdir(parents=True, exist_ok=True)
    # Resume from artifacts saved by an earlier interrupted session.
    for name in ("checkpoints", "results"):
        src = BACKUP / name
        dst = WORK / name
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
else:
    BACKUP = None

def backup_artifacts():
    if BACKUP is None:
        return
    for name in ("checkpoints", "results"):
        src = WORK / name
        if src.exists():
            shutil.copytree(src, BACKUP / name, dirs_exist_ok=True)
    print("Backed up artifacts to", BACKUP)


GPU: NVIDIA A100-SXM4-80GB | torch: 2.11.0+cu130
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — TweetQA training data

In [3]:
# Upload only the TweetQA training file. Validation is carved from this file.
# The test file is deliberately excluded from this development notebook.
name = "train.csv"
target = WORK / "data" / name
print("Expected training-data path:", target)
if target.exists():
    print("DATA BRANCH A: train.csv already exists at the expected path; no upload or copy needed.")
else:
    # VS Code's "Upload to Colab" normally puts the file in one of these locations.
    known_uploads = [
        Path("/content/train.csv"),
        Path("/content/data/train.csv"),
        Path("/content/A2/data/train.csv"),
    ]
    source = next((p for p in known_uploads if p.exists()), None)
    if source is not None:
        print("DATA BRANCH B: found a VS Code upload; copying", source, "->", target)
        shutil.copy(source, target)
    else:
        print("DATA BRANCH C: train.csv was not found; opening the browser upload widget.")
        from google.colab import files
        uploaded = files.upload()
        assert name in uploaded, f"Expected {name}; uploaded: {list(uploaded)}"
        target.write_bytes(uploaded[name])

import hashlib
expected = {"train.csv": "8a52bb7dff56b11fdad1ab6fd3b848fbadc22849"}
for name, sha1 in expected.items():
    # Git may check the CSV out with Windows CRLF endings. Normalise to LF so
    # byte-level line-ending changes do not reject otherwise identical data.
    content = (WORK / "data" / name).read_bytes().replace(b"\r\n", b"\n")
    actual = hashlib.sha1(content).hexdigest()
    print("SHA-1 check after newline normalisation:", name, actual)
    assert actual == sha1, f"Unexpected {name}; use the assignment-provided TweetQA file"
print("DATA READY: the training file is valid; continue to the next cell.")


Expected training-data path: /content/A2_v2/data/train.csv
DATA BRANCH A: train.csv already exists at the expected path; no upload or copy needed.
SHA-1 check after newline normalisation: train.csv 8a52bb7dff56b11fdad1ab6fd3b848fbadc22849
DATA READY: the training file is valid; continue to the next cell.


## Step 3 — Project source code

In [4]:
%%writefile src/__init__.py
# package marker


Overwriting src/__init__.py


In [5]:
%%writefile src/evaluate.py
"""Evaluation metrics for TweetQA.

Every metric is computed per example as the max over the gold answers,
and corpus scores are the mean x 100.

- BLEU-1: nltk sentence_bleu, weights (1,0,0,0), smoothing method1
- METEOR: nltk meteor_score (needs WordNet)
- ROUGE-L: our own LCS implementation, F-measure with beta = 1.2 (the
  coco-caption convention used by the official TweetQA evaluation)
- EM and token-F1: SQuAD-style, computed on normalised text
"""
import re
import string
import unicodedata
from collections import Counter

METRICS = ("bleu1", "meteor", "rougeL", "em", "f1")
_ARTICLES = re.compile(r"\b(a|an|the)\b")
_nltk_ready = False


def _ensure_nltk():
    global _nltk_ready
    if _nltk_ready:
        return
    import nltk
    for pkg, path in [("wordnet", "corpora/wordnet"), ("omw-1.4", "corpora/omw-1.4")]:
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(pkg, quiet=True)
    _nltk_ready = True


def normalize_answer(s):
    """Lowercase, replace ASCII and Unicode punctuation with spaces,
    drop articles, collapse whitespace."""
    s = s.lower()
    s = "".join(" " if (ch in string.punctuation or unicodedata.category(ch).startswith("P")) else ch
                for ch in s)
    s = _ARTICLES.sub(" ", s)
    return " ".join(s.split())


def norm_tokens(s):
    return normalize_answer(s).split()


# ---------------------------------------------------------------- metrics

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))


def token_f1(pred, gold):
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return float(p == g)
    common = Counter(p) & Counter(g)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    prec, rec = overlap / len(p), overlap / len(g)
    return 2 * prec * rec / (prec + rec)


def lcs_len(a, b):
    """Length of the longest common subsequence of two token lists (O(len(a)*len(b)) DP)."""
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0] * (len(b) + 1)
        for j, y in enumerate(b, 1):
            cur[j] = prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1])
        prev = cur
    return prev[-1]


def rouge_l(pred, gold, beta=1.2):
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return float(p == g and bool(p))
    lcs = lcs_len(p, g)
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(g)
    return (1 + beta ** 2) * prec * rec / (rec + beta ** 2 * prec)


def bleu1(pred, gold):
    from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return 0.0
    return sentence_bleu([g], p, weights=(1, 0, 0, 0),
                         smoothing_function=SmoothingFunction().method1)


def meteor(pred, gold):
    _ensure_nltk()
    from nltk.translate.meteor_score import meteor_score
    p, g = norm_tokens(pred), norm_tokens(gold)
    if not p or not g:
        return 0.0
    return meteor_score([g], p)


_FUNCS = {"bleu1": bleu1, "meteor": meteor, "rougeL": rouge_l, "em": exact_match, "f1": token_f1}


def score_example(pred, golds, metrics=METRICS):
    """Per-example scores in [0, 1]: max over gold answers."""
    pred = pred or ""
    return {m: max(_FUNCS[m](pred, g) for g in golds) for m in metrics}


def evaluate(preds, golds_list, metrics=METRICS, return_per_example=False):
    """preds: list[str]; golds_list: list[list[str]]. Returns corpus scores (mean x 100)."""
    assert len(preds) == len(golds_list), (len(preds), len(golds_list))
    per = [score_example(p, g, metrics) for p, g in zip(preds, golds_list)]
    n = max(1, len(per))
    corpus = {m: 100.0 * sum(s[m] for s in per) / n for m in metrics}
    return (corpus, per) if return_per_example else corpus


def fmt(scores):
    return " ".join(f"{m}={scores[m]:.2f}" for m in METRICS if m in scores)


Overwriting src/evaluate.py


In [6]:
%%writefile src/data.py
"""TweetQA data pipeline: loading, splits, offset-preserving tokenisation,
tweet normalisation, token flags, silver-span labelling, vocabularies and
a padded DataLoader.

Design notes (see results/DATA_REPORT.md):
- test.csv is the original TweetQA dev set. It stays locked (TEST_LOCKED)
  until the final evaluation. Validation is 10% of train, grouped by tweet.
- Every token keeps its character offsets into the raw tweet, so a
  predicted span (i, j) maps back to tweet[start_i:end_j].
- The normaliser only changes the *word* used for the embedding lookup.
  The char-CNN always sees the raw (lowercased) token, so @handles stay
  distinguishable after @user -> <user>.
"""
import ast
import csv
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset

try:
    from .evaluate import normalize_answer, rouge_l
except ImportError:  # inlined in the notebook / run as a script
    from evaluate import normalize_answer, rouge_l

TEST_LOCKED = True
DATA_DIR = Path(os.environ.get("TWEETQA_DIR", Path(__file__).resolve().parents[1] / "data"
                               if "__file__" in globals() else "data"))
SPLIT_SEED = 4012
VAL_FRAC = 0.10

# ------------------------------------------------------------------ loading

_ANSWER_ITEM = re.compile(r"'(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\"", re.S)


def parse_answers(field):
    """The Answer column is a numpy-array repr: ['a' 'b'] (space/newline separated, mixed quotes)."""
    items = [ast.literal_eval(m) for m in _ANSWER_ITEM.findall(field)]
    if not items:
        raise ValueError(f"unparseable answer field: {field!r}")
    return [str(a) for a in items]


def _read_csv(name):
    with open(Path(DATA_DIR) / name, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    return [dict(row_id=i, qid=r["qid"], question=r["Question"], tweet=r["Tweet"],
                 answers=parse_answers(r["Answer"]), answer_field=r["Answer"])
            for i, r in enumerate(rows)]


def _question_word(q):
    toks = tokenize(q.lower())
    w = toks[0][0] if toks else ""
    return w if w in {"what", "who", "how", "where", "why", "which", "when"} else "other"


def get_splits(seed=SPLIT_SEED, val_frac=VAL_FRAC):
    """Train/val from train.csv: exact duplicates dropped, grouped by tweet,
    stratified by the first question word of each tweet's first question."""
    rows, seen = [], set()
    for r in _read_csv("train.csv"):
        key = (r["question"], r["answer_field"], r["tweet"])
        if key not in seen:
            seen.add(key)
            rows.append(r)
    by_tweet = defaultdict(list)
    for r in rows:
        by_tweet[r["tweet"]].append(r)
    strata = defaultdict(list)
    for tweet in sorted(by_tweet):
        strata[_question_word(by_tweet[tweet][0]["question"])].append(tweet)
    rng = random.Random(seed)
    val_tweets = set()
    for s in sorted(strata):
        tweets = strata[s][:]
        rng.shuffle(tweets)
        val_tweets.update(tweets[:round(len(tweets) * val_frac)])
    train = [r for r in rows if r["tweet"] not in val_tweets]
    val = [r for r in rows if r["tweet"] in val_tweets]
    return train, val


def load_test():
    if TEST_LOCKED:
        raise RuntimeError("Test set is locked (TEST_LOCKED=True). It is used once, in Stage 8.")
    return _read_csv("test.csv")


# ------------------------------------------------------------- tokenisation

_URL = r"https?://\S+|www\.\S+"
_EMOJI_CHARS = "\U0001F000-\U0001FAFF☀-➿⬀-⯿️‍"
_TOKEN_RE = re.compile(
    rf"(?P<url>{_URL})"
    r"|(?P<mention>@\w+)"
    r"|(?P<hashtag>#\w+)"
    rf"|(?P<emoji>[{_EMOJI_CHARS}]+)"
    r"|(?P<number>\d+(?:[.,:/]\d+)*)"
    r"|(?P<word>\w+(?:['’]\w+)*)"
    r"|(?P<sym>[^\w\s])",
    re.UNICODE,
)
_ELONG = re.compile(r"(.)\1{2,}")


def tokenize(text):
    """Returns [(raw_token, start, end, kind)] with char offsets into text."""
    return [(m.group(0), m.start(), m.end(), m.lastgroup) for m in _TOKEN_RE.finditer(text)]


def normalise_tokens(raw_tokens, use_normaliser=True):
    """Map raw tokens to model tokens: dicts with word, raw, start, end, kind.

    Normaliser: @user -> <user>, url -> <url>, emoji -> <emoji>,
    #tag -> '#' + 'tag', elongation squashed to 2 repeats, lowercase.
    With use_normaliser=False the word is just the lowercased raw token.
    """
    out = []
    for raw, s, e, kind in raw_tokens:
        if not use_normaliser:
            out.append(dict(word=raw.lower(), raw=raw, start=s, end=e, kind=kind))
        elif kind == "mention":
            out.append(dict(word="<user>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "url":
            out.append(dict(word="<url>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "emoji":
            out.append(dict(word="<emoji>", raw=raw, start=s, end=e, kind=kind))
        elif kind == "hashtag":
            out.append(dict(word="#", raw="#", start=s, end=s + 1, kind=kind))
            tag = raw[1:]
            out.append(dict(word=_ELONG.sub(r"\1\1", tag.lower()), raw=tag, start=s + 1, end=e, kind=kind))
        else:
            out.append(dict(word=_ELONG.sub(r"\1\1", raw.lower()), raw=raw, start=s, end=e, kind=kind))
    return out


_BYLINE_RE = re.compile(r"\(@\w+\)\s+[A-Za-z]+\.?\s+\d{1,2},\s+\d{4}\s*$")
_BOUNDARY = {".", "!", "?", "—", "–", "-", "\"", "”", "|", ":"}


def byline_start(text, tokens):
    """Index of the first token of the trailing 'Name (@handle) Month D, YYYY'
    byline, or len(tokens) if there is none. The name is taken back to the
    previous sentence boundary, at most 6 tokens."""
    m = _BYLINE_RE.search(text)
    if not m:
        return len(tokens)
    k = next((i for i, t in enumerate(tokens) if t["start"] >= m.start()), len(tokens))
    j = k
    while j > 0 and k - j < 6 and tokens[j - 1]["raw"] not in _BOUNDARY:
        j -= 1
    return j


FLAG_NAMES = ("mention", "hashtag", "url", "emoji", "number", "capitalised", "in_byline", "in_other_seq")


def token_flags(tokens, other_words, byline_from=None):
    by = len(tokens) if byline_from is None else byline_from
    flags = []
    for i, t in enumerate(tokens):
        flags.append([
            float(t["kind"] == "mention"), float(t["kind"] == "hashtag"), float(t["kind"] == "url"),
            float(t["kind"] == "emoji"), float(t["kind"] == "number"),
            float(t["raw"][:1].isupper()), float(i >= by),
            float(t["word"] in other_words and t["word"] not in _STOP),
        ])
    return flags


_STOP = set("""a an the of to in on at for by with from and or but is are was were be been being am
do does did has have had i you he she it we they me him her us them my your his its our their this that
these those what who whom whose which when where why how not no yes so if than then there here as just
about into out up down over can will would should could may might must 's ? . , ! ' " - — : ; ( )""".split())


# ------------------------------------------------------------- silver spans

def _norm_toks(raw):
    return normalize_answer(raw).split()


def silver_spans(tokens, tweet, answers, max_len=15, tau=0.5):
    """Score every span (i, j), j - i < max_len, that starts and ends on a
    content token, by token-F1 against each gold answer (max over golds).

    Returns (best, best_f1, soft):
      best  : (i, j) with the highest F1, ties broken by ROUGE-L then length
      soft  : [(i, j, w)] for all spans with F1 >= tau, weights proportional to F1
    """
    ntok = [_norm_toks(t["raw"]) for t in tokens]
    content = [bool(x) for x in ntok]
    golds = [Counter(_norm_toks(a)) for a in answers]
    glens = [sum(g.values()) for g in golds]
    scored = []
    for i in range(len(tokens)):
        if not content[i]:
            continue
        bag, n = Counter(), 0
        for j in range(i, min(len(tokens), i + max_len)):
            bag.update(ntok[j])
            n += len(ntok[j])
            if not content[j]:
                continue
            f = 0.0
            for g, gl in zip(golds, glens):
                if gl == 0:
                    continue
                ov = sum((bag & g).values())
                if ov:
                    p, r = ov / n, ov / gl
                    f = max(f, 2 * p * r / (p + r))
            if f > 0:
                scored.append((f, i, j))
    if not scored:
        return None, 0.0, []
    top = max(f for f, _, _ in scored)
    ties = [(i, j) for f, i, j in scored if f == top]
    if len(ties) > 1:
        def key(ij):
            txt = tweet[tokens[ij[0]]["start"]:tokens[ij[1]]["end"]]
            return (max(rouge_l(txt, a) for a in answers), -(ij[1] - ij[0]), -ij[0])
        ties.sort(key=key, reverse=True)
    soft = [(i, j, f) for f, i, j in scored if f >= tau]
    z = sum(w for _, _, w in soft)
    soft = [(i, j, w / z) for i, j, w in soft]
    return ties[0], top, soft


def span_text(ex, i, j):
    return ex["tweet"][ex["t_tokens"][i]["start"]:ex["t_tokens"][j]["end"]]


# ------------------------------------------------------------------- vocab

class Vocab:
    PAD, UNK = "<pad>", "<unk>"

    def __init__(self, counter, min_freq=2, specials=()):
        self.itos = [self.PAD, self.UNK] + [s for s in specials if s not in (self.PAD, self.UNK)]
        self.itos += sorted([w for w, c in counter.items() if c >= min_freq and w not in self.itos],
                            key=lambda w: (-counter[w], w))
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def __call__(self, w):
        return self.stoi.get(w, 1)


SPECIALS = ("<user>", "<url>", "<emoji>")
MAX_WORD_CHARS = 16


def build_vocabs(examples, min_freq=2, char_min_freq=5):
    """Vocabularies come from the training split only."""
    wc, cc = Counter(), Counter()
    for ex in examples:
        for t in ex["t_tokens"] + ex["q_tokens"]:
            wc[t["word"]] += 1
            cc.update(t["raw"].lower()[:MAX_WORD_CHARS])
    return Vocab(wc, min_freq, SPECIALS), Vocab(cc, char_min_freq)


# ----------------------------------------------------------------- encoding

# tau: an example is trainable if its best silver span has F1 >= tau.
# soft_tau: minimum F1 for a span to join the soft target set (None = same as tau). Kept separate so the
# soft-set width can be varied without changing which examples are trained on.
DEFAULT_PREP = dict(use_normaliser=True, max_span_len=15, tau=0.5, soft_tau=None)


def preprocess(rows, prep=None):
    """Tokenise, normalise, flag and silver-label raw rows."""
    prep = {**DEFAULT_PREP, **(prep or {})}
    out = []
    for r in rows:
        t_tokens = normalise_tokens(tokenize(r["tweet"]), prep["use_normaliser"])
        q_tokens = normalise_tokens(tokenize(r["question"]), prep["use_normaliser"])
        by = byline_start(r["tweet"], t_tokens)
        t_words = {t["word"] for t in t_tokens}
        q_words = {t["word"] for t in q_tokens}
        soft_tau = prep["tau"] if prep.get("soft_tau") is None else prep["soft_tau"]
        best, best_f1, soft = silver_spans(t_tokens, r["tweet"], r["answers"], prep["max_span_len"], soft_tau)
        out.append(dict(
            r, t_tokens=t_tokens, q_tokens=q_tokens, byline_from=by,
            t_flags=token_flags(t_tokens, q_words, by), q_flags=token_flags(q_tokens, t_words),
            best=best, best_f1=best_f1, soft=soft,
            trainable=best is not None and best_f1 >= prep["tau"],
        ))
    return out


def encode(examples, wvocab, cvocab):
    for ex in examples:
        for side in ("t", "q"):
            toks = ex[f"{side}_tokens"]
            ex[f"{side}_ids"] = [wvocab(t["word"]) for t in toks]
            ex[f"{side}_chars"] = [[cvocab(c) for c in t["raw"].lower()[:MAX_WORD_CHARS]] or [1] for t in toks]
    return examples


class QADataset(Dataset):
    def __init__(self, examples, train=False):
        # during training, only examples with a usable silver span contribute to the loss
        self.examples = [e for e in examples if e["trainable"]] if train else list(examples)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


def _pad_side(batch, side, n_flags):
    B = len(batch)
    T = max(1, max(len(ex[f"{side}_ids"]) for ex in batch))
    W = max(1, max((len(c) for ex in batch for c in ex[f"{side}_chars"]), default=1))
    words = torch.zeros(B, T, dtype=torch.long)
    chars = torch.zeros(B, T, W, dtype=torch.long)
    flags = torch.zeros(B, T, n_flags)
    mask = torch.zeros(B, T, dtype=torch.bool)
    for b, ex in enumerate(batch):
        n = len(ex[f"{side}_ids"])
        words[b, :n] = torch.tensor(ex[f"{side}_ids"], dtype=torch.long)
        for k, cs in enumerate(ex[f"{side}_chars"]):
            chars[b, k, :len(cs)] = torch.tensor(cs, dtype=torch.long)
        if n:
            flags[b, :n] = torch.tensor(ex[f"{side}_flags"])
        mask[b, :n] = True
    return words, chars, flags, mask


def collate(batch):
    tw, tc, tf, tm = _pad_side(batch, "t", len(FLAG_NAMES))
    qw, qc, qf, qm = _pad_side(batch, "q", len(FLAG_NAMES))
    B, T = tw.shape
    y_start = torch.full((B,), -1, dtype=torch.long)
    y_end = torch.full((B,), -1, dtype=torch.long)
    soft_start = torch.zeros(B, T)
    soft_end = torch.zeros(B, T)
    for b, ex in enumerate(batch):
        if ex["best"] is not None:
            y_start[b], y_end[b] = ex["best"]
        for i, j, w in ex["soft"]:
            soft_start[b, i] += w
            soft_end[b, j] += w
    return dict(t_words=tw, t_chars=tc, t_flags=tf, t_mask=tm,
                q_words=qw, q_chars=qc, q_flags=qf, q_mask=qm,
                y_start=y_start, y_end=y_end, soft_start=soft_start, soft_end=soft_end,
                meta=batch)


def make_loader(examples, batch_size=32, train=False, seed=0):
    ds = QADataset(examples, train=train)
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=train, collate_fn=collate, generator=g)


# ------------------------------------------------------------------ oracle

def oracle_predictions(examples):
    """Best silver span text for every example ('' if there is no overlapping span)."""
    return [span_text(ex, *ex["best"]) if ex["best"] is not None else "" for ex in examples]


Overwriting src/data.py


In [7]:
%%writefile src/runlog.py
"""Append-only experiment log: every reported number must come from a row in results/runs.csv."""
import csv
import json
import os
import time
from pathlib import Path

RESULTS_DIR = Path(os.environ.get("A2_RESULTS", Path(__file__).resolve().parents[1] / "results"
                                  if "__file__" in globals() else "results"))
RUNS_CSV = RESULTS_DIR / "runs.csv"
FIELDS = ["run_id", "timestamp", "model", "config", "seed", "split",
          "bleu1", "meteor", "rougeL", "em", "f1", "train_time_s", "notes"]


def log_run(model, config, seed, split, scores, train_time_s=0.0, run_id=None, notes=""):
    RUNS_CSV.parent.mkdir(parents=True, exist_ok=True)
    new = not RUNS_CSV.exists()
    run_id = run_id or f"{model}-{split}-s{seed}-{time.strftime('%Y%m%d-%H%M%S')}"
    row = dict(run_id=run_id, timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), model=model,
               config=json.dumps(config, sort_keys=True, default=str), seed=seed, split=split,
               train_time_s=round(train_time_s, 1), notes=notes,
               **{m: round(scores[m], 4) for m in ("bleu1", "meteor", "rougeL", "em", "f1")})
    with open(RUNS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    return run_id


def read_runs():
    if not RUNS_CSV.exists():
        return []
    with open(RUNS_CSV, encoding="utf-8") as f:
        return list(csv.DictReader(f))


Overwriting src/runlog.py


In [8]:
%%writefile src/model.py
"""TweetBiDAF: an extractive QA model for tweets, trained from scratch.

  1. Embedding: word embedding ⊕ char-CNN over the *raw* token ⊕ token flags
     -> 2-layer highway                                    (A1: use_char_cnn, A2: use_token_flags)
  2. Shared BiLSTM contextual encoder for tweet (H) and question (U)
  3. Trilinear similarity S_ij = w^T [h_i; u_j; h_i ∘ u_j]
     C2Q = softmax_j(S) U,  Q2C = softmax_i(max_j S) H
     G = [h; c2q; h ∘ c2q; h ∘ q2c]                        (A4: biattn_mode "c2q" drops Q2C)
  4. Gated multi-head self-attention over the projected G,
     residual + LayerNorm                                   (A3: use_self_attn)
  5. 2-layer modelling BiLSTM, start head, 1 more BiLSTM layer, end head,
     padding masks, best (i, j) with i <= j < i + L

The design follows BiDAF (Seo et al., 2017) with gated self-attention in the
style of R-Net (Wang et al., 2017). The tweet-specific parts are written for
this project: a char-CNN on raw tokens with a normalised word vocabulary,
token flags (including the byline flag), soft silver-span training, and
self-attention for long-range links between the tweet body and its byline.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

NEG = -1e30

DEFAULT_MODEL = dict(
    word_dim=100, char_dim=32, char_filters=50, char_widths=(3, 4, 5), n_flags=8,
    hidden=128, dropout=0.2, attn_heads=4, max_span_len=15,
    use_char_cnn=True, use_token_flags=True, use_self_attn=True, biattn_mode="both",
)


class CharCNN(nn.Module):
    def __init__(self, n_chars, char_dim, filters, widths):
        super().__init__()
        self.emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.convs = nn.ModuleList(nn.Conv1d(char_dim, filters, w, padding=w // 2) for w in widths)
        self.out_dim = filters * len(widths)

    def forward(self, chars):                      # (B, T, W)
        B, T, W = chars.shape
        x = self.emb(chars.view(B * T, W)).transpose(1, 2)            # (BT, c, W)
        cmask = (chars.view(B * T, 1, W) != 0)
        outs = []
        for conv in self.convs:
            y = F.relu(conv(x))[:, :, :W]
            y = y.masked_fill(~cmask, 0.0)          # ReLU output >= 0, so masked max-pool is safe
            outs.append(y.max(dim=2).values)
        return torch.cat(outs, -1).view(B, T, -1)


class Highway(nn.Module):
    def __init__(self, dim, n_layers=2):
        super().__init__()
        self.lin = nn.ModuleList(nn.Linear(dim, dim) for _ in range(n_layers))
        self.gate = nn.ModuleList(nn.Linear(dim, dim) for _ in range(n_layers))

    def forward(self, x):
        for lin, gate in zip(self.lin, self.gate):
            g = torch.sigmoid(gate(x))
            x = g * F.relu(lin(x)) + (1 - g) * x
        return x


def reverse_padded(x, mask):
    """Reverse each sequence within its own length; padding stays at the end."""
    T = x.size(1)
    lengths = mask.sum(1, keepdim=True)                                  # (B, 1)
    t = torch.arange(T, device=x.device).unsqueeze(0)                    # (1, T)
    idx = torch.where(t < lengths, lengths - 1 - t, t)                   # (B, T)
    return x.gather(1, idx.unsqueeze(-1).expand_as(x))


class BiLSTM(nn.Module):
    """BiLSTM over right-padded batches, padding-exact without packing.

    The forward LSTM reads valid tokens before any padding, so its outputs at
    valid positions are exact. The backward direction runs a forward LSTM over
    each sequence reversed *within its length*, then un-reverses the output.
    The result equals a packed BiLSTM, but runs ~4x faster on MPS.
    """

    def __init__(self, in_dim, hidden, layers=1, dropout=0.0):
        super().__init__()
        self.fw = nn.ModuleList(nn.LSTM(in_dim if l == 0 else 2 * hidden, hidden, batch_first=True)
                                for l in range(layers))
        self.bw = nn.ModuleList(nn.LSTM(in_dim if l == 0 else 2 * hidden, hidden, batch_first=True)
                                for l in range(layers))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        for l, (fw, bw) in enumerate(zip(self.fw, self.bw)):
            if l > 0:
                x = self.drop(x)
            f = fw(x)[0]
            b = reverse_padded(bw(reverse_padded(x, mask))[0], mask)
            x = torch.cat([f, b], -1) * mask.unsqueeze(-1)
        return x


class GatedSelfAttention(nn.Module):
    """Multi-head self-attention, then a sigmoid gate on [x; attended], then residual + LayerNorm."""

    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.mha = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.gate = nn.Linear(2 * dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask):
        attended, w = self.mha(x, x, x, key_padding_mask=~mask,
                               need_weights=True, average_attn_weights=True)
        g = torch.sigmoid(self.gate(torch.cat([x, attended], -1)))
        return self.norm(x + self.drop(g * attended)), w


class TweetBiDAF(nn.Module):
    def __init__(self, n_words, n_chars, cfg=None, word_vectors=None):
        super().__init__()
        self.cfg = cfg = {**DEFAULT_MODEL, **(cfg or {})}
        H, p = cfg["hidden"], cfg["dropout"]
        self.word_emb = nn.Embedding(n_words, cfg["word_dim"], padding_idx=0)
        if word_vectors is not None:
            self.word_emb.weight.data.copy_(word_vectors)
        emb_dim = cfg["word_dim"]
        if cfg["use_char_cnn"]:
            self.char_cnn = CharCNN(n_chars, cfg["char_dim"], cfg["char_filters"], cfg["char_widths"])
            emb_dim += self.char_cnn.out_dim
        if cfg["use_token_flags"]:
            emb_dim += cfg["n_flags"]
        self.highway = Highway(emb_dim, 2)
        self.drop = nn.Dropout(p)
        self.encoder = BiLSTM(emb_dim, H)
        self.w_sim = nn.Linear(6 * H, 1)           # trilinear: w^T [h; u; h∘u]
        g_dim = 8 * H if cfg["biattn_mode"] == "both" else 6 * H
        self.g_proj = nn.Linear(g_dim, 2 * H)
        if cfg["use_self_attn"]:
            self.self_attn = GatedSelfAttention(2 * H, cfg["attn_heads"], p)
        self.model1 = BiLSTM(2 * H, H, layers=2, dropout=p)
        self.model2 = BiLSTM(2 * H, H)
        self.start_head = nn.Linear(4 * H, 1)
        self.end_head = nn.Linear(4 * H, 1)

    # ---------------------------------------------------------------- layers
    def embed(self, words, chars, flags):
        parts = [self.word_emb(words)]
        if self.cfg["use_char_cnn"]:
            parts.append(self.char_cnn(chars))
        if self.cfg["use_token_flags"]:
            parts.append(flags)
        return self.highway(self.drop(torch.cat(parts, -1)))

    def similarity(self, h, u):
        """S_ij = w_h·h_i + w_u·u_j + w_hu·(h_i∘u_j), computed without materialising [h; u; h∘u]."""
        d = h.size(-1)
        w = self.w_sim.weight.squeeze(0)
        wh, wu, whu = w[:d], w[d:2 * d], w[2 * d:]
        return (h @ wh).unsqueeze(2) + (u @ wu).unsqueeze(1) + (h * whu) @ u.transpose(1, 2) + self.w_sim.bias

    def forward(self, batch, return_attn=False):
        tm, qm = batch["t_mask"], batch["q_mask"]
        h = self.drop(self.encoder(self.embed(batch["t_words"], batch["t_chars"], batch["t_flags"]), tm))
        u = self.drop(self.encoder(self.embed(batch["q_words"], batch["q_chars"], batch["q_flags"]), qm))

        S = self.similarity(h, u)                                        # (B, T, J)
        a = F.softmax(S.masked_fill(~qm.unsqueeze(1), NEG), dim=-1)      # C2Q weights
        c2q = a @ u
        parts = [h, c2q, h * c2q]
        b = None
        if self.cfg["biattn_mode"] == "both":
            smax = S.masked_fill(~qm.unsqueeze(1), NEG).max(-1).values    # (B, T)
            b = F.softmax(smax.masked_fill(~tm, NEG), dim=-1)            # Q2C weights
            q2c = (b.unsqueeze(1) @ h).expand_as(h)
            parts.append(h * q2c)
        x = self.drop(F.relu(self.g_proj(torch.cat(parts, -1))))

        sa = None
        if self.cfg["use_self_attn"]:
            x, sa = self.self_attn(x, tm)

        m1 = self.drop(self.model1(x, tm))
        m2 = self.drop(self.model2(m1, tm))
        start = self.start_head(torch.cat([x, m1], -1)).squeeze(-1).masked_fill(~tm, NEG)
        end = self.end_head(torch.cat([x, m2], -1)).squeeze(-1).masked_fill(~tm, NEG)
        out = dict(start_logits=start, end_logits=end)
        if return_attn:
            out.update(S=S, c2q=a, q2c=b, self_attn=sa)
        return out


# ------------------------------------------------------------------ loss / decode

def span_loss(out, batch, soft=True):
    ls = F.log_softmax(out["start_logits"], -1)
    le = F.log_softmax(out["end_logits"], -1)
    if soft:
        # soft cross-entropy against the marginal start/end distributions of the soft silver set
        return -(batch["soft_start"] * ls).sum(-1).mean() - (batch["soft_end"] * le).sum(-1).mean()
    return F.nll_loss(ls, batch["y_start"]) + F.nll_loss(le, batch["y_end"])


@torch.no_grad()
def decode(start_logits, end_logits, max_len=15):
    """argmax over i <= j < i + max_len of log p_s(i) + log p_e(j). Returns (B, 2) indices and scores."""
    ls = F.log_softmax(start_logits, -1)
    le = F.log_softmax(end_logits, -1)
    T = ls.size(1)
    scores = ls.unsqueeze(2) + le.unsqueeze(1)                           # (B, T, T)
    idx = torch.arange(T, device=ls.device)
    band = (idx.unsqueeze(0) >= idx.unsqueeze(1)) & (idx.unsqueeze(0) - idx.unsqueeze(1) < max_len)
    scores = scores.masked_fill(~band, NEG)
    flat = scores.view(scores.size(0), -1)
    best = flat.argmax(-1)
    return torch.stack([best // T, best % T], -1), flat.max(-1).values


Overwriting src/model.py


In [9]:
%%writefile src/fasttext_init.py
"""Optional in-domain word vectors: FastText (gensim) trained from scratch on the
*training split's* tweets and questions only. No external text and no
pretrained weights, so the main model is still trained from scratch on
TweetQA alone."""
import torch


def fasttext_vectors(wv, cfg, train_examples, epochs=20, seed=1):
    from gensim.models import FastText
    sents, seen = [], set()
    for ex in train_examples:
        if ex["tweet"] not in seen:                 # each tweet once
            seen.add(ex["tweet"])
            sents.append([t["word"] for t in ex["t_tokens"]])
        sents.append([t["word"] for t in ex["q_tokens"]])
    ft = FastText(vector_size=cfg["word_dim"], window=5, min_count=1, sg=1, epochs=epochs,
                  seed=seed, workers=1, min_n=3, max_n=5)
    ft.build_vocab(sents)
    ft.train(sents, total_examples=len(sents), epochs=epochs)
    W = torch.randn(len(wv), cfg["word_dim"]) * 0.1
    W[0] = 0.0
    for i, w in enumerate(wv.itos[2:], start=2):
        W[i] = torch.tensor(ft.wv[w])               # subword n-grams also give OOV-safe vectors
    return W


Overwriting src/fasttext_init.py


In [10]:
%%writefile src/train.py
"""Training loop for TweetBiDAF: config dict -> train / early stop / checkpoint / runs.csv.

Usage (from A2/):
  python -m src.train --name main --seed 1                        # full run
  python -m src.train --name smoke --smoke                        # 200 examples, 2 epochs
  python -m src.train --name a3 --set use_self_attn=false --seed 2
"""
import argparse
import json
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch

try:
    from . import data as D
    from .evaluate import evaluate, fmt
    from .model import TweetBiDAF, decode, span_loss
    from .runlog import RESULTS_DIR, log_run
except ImportError:
    import data as D
    from evaluate import evaluate, fmt
    from model import TweetBiDAF, decode, span_loss
    from runlog import RESULTS_DIR, log_run

DEFAULT_TRAIN = dict(
    lr=1e-3, batch_size=32, epochs=30, patience=4, grad_clip=5.0, weight_decay=0.0,
    use_normaliser=True, tau=0.5, soft_tau=None, soft_spans=True, min_freq=2, word_init="random",
    max_span_len=15,
)
CKPT_DIR = Path(os.environ.get("A2_CKPT", RESULTS_DIR.parent / "checkpoints"))


def get_device():
    return "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)


def to_device(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def prepare_data(cfg, train_rows=None, val_rows=None, limit=None):
    """Preprocess and encode the train/val splits. Vocab comes from the training split only."""
    if train_rows is None:
        train_rows, val_rows = D.get_splits()
    if limit:
        train_rows, val_rows = train_rows[:limit], val_rows[:limit]
    prep = dict(use_normaliser=cfg["use_normaliser"], max_span_len=cfg["max_span_len"], tau=cfg["tau"],
                soft_tau=cfg.get("soft_tau"))
    tr, va = D.preprocess(train_rows, prep), D.preprocess(val_rows, prep)
    wv, cv = D.build_vocabs(tr, min_freq=cfg["min_freq"])
    D.encode(tr, wv, cv)
    D.encode(va, wv, cv)
    return tr, va, wv, cv


@torch.no_grad()
def predict(model, examples, device, batch_size=64, max_span_len=15):
    model.eval()
    preds, spans = [], []
    for batch in D.make_loader(examples, batch_size=batch_size):
        meta = batch["meta"]
        out = model(to_device(batch, device))
        idx, _ = decode(out["start_logits"], out["end_logits"], max_span_len)
        for ex, (i, j) in zip(meta, idx.tolist()):
            spans.append((i, j))
            preds.append(D.span_text(ex, i, j))
    return preds, spans


def build_model(cfg, wv, cv, word_vectors=None):
    mcfg = {k: cfg[k] for k in _default_model() if k in cfg}
    return TweetBiDAF(len(wv), len(cv), mcfg, word_vectors)


def train(cfg=None, seed=1, name="main", limit=None, data=None, log=print, save=True):
    """Train one model. Returns dict(best scores, history, checkpoint path, run_id)."""
    cfg = {**DEFAULT_TRAIN, **_default_model(), **(cfg or {})}
    seed_everything(seed)
    device = get_device()
    tr, va, wv, cv = data if data is not None else prepare_data(cfg, limit=limit)
    wvecs = None
    if cfg["word_init"] == "fasttext":
        try:
            from .fasttext_init import fasttext_vectors
        except ImportError:
            from fasttext_init import fasttext_vectors
        wvecs = fasttext_vectors(wv, cfg, tr, seed=seed)
    model = build_model(cfg, wv, cv, wvecs).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    loader = D.make_loader(tr, batch_size=cfg["batch_size"], train=True, seed=seed)
    golds = [e["answers"] for e in va]
    run_id = f"{name}-s{seed}-{time.strftime('%Y%m%d-%H%M%S')}"
    log(f"[{run_id}] device={device} params={n_params:,} train_used={len(loader.dataset)}/{len(tr)} "
        f"val={len(va)} vocab={len(wv)} chars={len(cv)}")
    log(f"[{run_id}] config={json.dumps(cfg, sort_keys=True)}")

    best, best_epoch, bad, history, best_preds = None, -1, 0, [], []
    ckpt = CKPT_DIR / f"{run_id}.pt"
    t_start = time.time()
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        t0, tot, n = time.time(), 0.0, 0
        for batch in loader:
            batch = to_device(batch, device)
            loss = span_loss(model(batch), batch, soft=cfg["soft_spans"])
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            opt.step()
            tot += loss.item() * batch["t_words"].size(0)
            n += batch["t_words"].size(0)
        train_time = time.time() - t0
        preds, spans = predict(model, va, device, max_span_len=cfg["max_span_len"])
        scores = evaluate(preds, golds)
        history.append(dict(epoch=epoch, train_loss=tot / max(1, n), epoch_time=train_time, **scores))
        improved = best is None or scores["rougeL"] > best["rougeL"]
        log(f"[{run_id}] epoch {epoch:2d} loss={tot / max(1, n):.4f} time={train_time:.1f}s val {fmt(scores)}"
            + (" *" if improved else ""))
        if improved:
            best, best_epoch, bad = scores, epoch, 0
            best_preds = [dict(row_id=e["row_id"], qid=e["qid"], pred=p, span=sp)
                          for e, p, sp in zip(va, preds, spans)]
            if save:
                CKPT_DIR.mkdir(parents=True, exist_ok=True)
                torch.save(dict(state_dict=model.state_dict(), cfg=cfg, seed=seed, epoch=epoch,
                                vocab=wv.itos, chars=cv.itos), ckpt)
        else:
            bad += 1
            if bad >= cfg["patience"]:
                log(f"[{run_id}] early stop at epoch {epoch} (best epoch {best_epoch})")
                break
    total = time.time() - t_start
    log(f"[{run_id}] best epoch {best_epoch}: val {fmt(best)} | total {total/60:.1f} min")
    if save:   # smoke tests (save=False) leave no files behind, so results/ only holds logged runs
        hist_dir = RESULTS_DIR / "history"
        hist_dir.mkdir(parents=True, exist_ok=True)
        (hist_dir / f"{run_id}.json").write_text(json.dumps(dict(cfg=cfg, seed=seed, history=history,
                                                                 best_epoch=best_epoch), indent=1))
        pred_dir = RESULTS_DIR / "preds"
        pred_dir.mkdir(parents=True, exist_ok=True)
        (pred_dir / f"{run_id}.json").write_text(json.dumps(best_preds))
        log_run(name, cfg, seed, "val", best, total, run_id=run_id,
                notes=f"best_epoch={best_epoch} params={n_params} limit={limit} device={device}"
                + (f":{torch.cuda.get_device_name(0)}" if device == "cuda" else ""))
    return dict(best=best, best_epoch=best_epoch, history=history, ckpt=str(ckpt), run_id=run_id,
                model=model, data=(tr, va, wv, cv))


def _default_model():
    try:
        from .model import DEFAULT_MODEL
    except ImportError:
        from model import DEFAULT_MODEL
    return dict(DEFAULT_MODEL)


def load_checkpoint(path, device=None):
    device = device or get_device()
    ck = torch.load(path, map_location="cpu", weights_only=False)
    wv, cv = D.Vocab({}, 1), D.Vocab({}, 1)
    wv.itos, cv.itos = ck["vocab"], ck["chars"]
    wv.stoi = {w: i for i, w in enumerate(wv.itos)}
    cv.stoi = {c: i for i, c in enumerate(cv.itos)}
    model = build_model(ck["cfg"], wv, cv)
    model.load_state_dict(ck["state_dict"])
    return model.to(device).eval(), ck["cfg"], wv, cv


def _parse_set(items):
    """Parse key=json_value overrides. Unknown keys and values whose type doesn't match the default
    (e.g. a bool flag given a string) are errors, so a typo can never silently change an experiment."""
    defaults = {**DEFAULT_TRAIN, **_default_model()}
    out = {}
    for it in items or []:
        k, v = it.split("=", 1)
        if k not in defaults:
            raise SystemExit(f"--set: unknown config key {k!r}")
        try:
            val = json.loads(v)
        except json.JSONDecodeError:
            val = v
        d = defaults[k]
        ok = (d is None or val is None or isinstance(val, type(d))
              or (isinstance(d, float) and isinstance(val, int) and not isinstance(val, bool))
              or (isinstance(d, tuple) and isinstance(val, list)))
        if isinstance(d, bool) and not isinstance(val, bool):
            ok = False
        if not ok:
            raise SystemExit(f"--set: {k}={v!r} has type {type(val).__name__}, expected {type(d).__name__}")
        out[k] = val
    return out


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", default="main")
    ap.add_argument("--seed", type=int, default=1)
    ap.add_argument("--smoke", action="store_true", help="200 examples, 2 epochs, no runs.csv entry")
    ap.add_argument("--set", nargs="*", help="config overrides key=json_value")
    args = ap.parse_args()
    cfg = _parse_set(args.set)
    if args.smoke:
        cfg.update(epochs=2, patience=99)
    res = train(cfg, seed=args.seed, name=args.name, limit=200 if args.smoke else None,
                log=lambda s: print(s, flush=True), save=not args.smoke)
    if args.smoke:
        h = res["history"]
        assert h[-1]["train_loss"] < h[0]["train_loss"], "smoke test: loss did not decrease"
        tr, va, wv, cv = res["data"]
        model = res["model"]
        batch = to_device(D.collate(va[:4]), get_device())
        with torch.no_grad():
            model.eval()
            o = model(batch, return_attn=True)
        B, T = batch["t_words"].shape
        J = batch["q_words"].shape[1]
        assert o["S"].shape == (B, T, J) and o["c2q"].shape == (B, T, J)
        if o["q2c"] is not None:
            assert o["q2c"].shape == (B, T)
        if o["self_attn"] is not None:
            assert o["self_attn"].shape == (B, T, T)
        assert torch.allclose(o["c2q"].sum(-1), torch.ones(B, T, device=o["c2q"].device), atol=1e-4)
        print(f"SMOKE OK: loss {h[0]['train_loss']:.3f} -> {h[-1]['train_loss']:.3f}; "
              f"attn shapes S{tuple(o['S'].shape)} self{tuple(o['self_attn'].shape) if o['self_attn'] is not None else None}")
    sys.exit(0)


Overwriting src/train.py


In [11]:
%%writefile src/analysis.py
"""Analysis utilities: hypothesis slices, ablation tables, attention heatmaps, error sampling."""
import csv
import json
import math
import random
import statistics
from collections import defaultdict
from pathlib import Path

import torch

try:
    from . import data as D
    from .evaluate import METRICS, evaluate, normalize_answer, score_example
    from .runlog import RESULTS_DIR, read_runs
except ImportError:
    import data as D
    from evaluate import METRICS, evaluate, normalize_answer, score_example
    from runlog import RESULTS_DIR, read_runs

FIG_DIR = RESULTS_DIR / "figures"
TAB_DIR = RESULTS_DIR / "tables"
FAR_N = 8   # A3 slice: answer more than FAR_N tokens from the nearest question-word match


# ------------------------------------------------------------------ slices

def _answer_far_or_byline(ex):
    if ex["best"] is None:
        return False
    i, j = ex["best"]
    if i >= ex["byline_from"]:
        return True
    qwords = {w for t in ex["q_tokens"] for w in normalize_answer(t["raw"]).split()} - D._STOP
    anchors = [p for p, t in enumerate(ex["t_tokens"])
               if set(normalize_answer(t["raw"]).split()) & qwords and not (i <= p <= j)]
    if not anchors:
        return True
    return min(i - p if p < i else p - j for p in anchors) > FAR_N


def _answer_is_handle_or_tag(ex):
    if ex["best"] is None:
        return any(a.strip().startswith(("@", "#")) for a in ex["answers"])
    i, j = ex["best"]
    toks = ex["t_tokens"][i:j + 1]
    return (any(t["kind"] in ("mention", "hashtag") for t in toks) or i >= ex["byline_from"]
            or any(a.strip().startswith(("@", "#")) for a in ex["answers"]))


def slices(examples, wvocab):
    """row_id sets for the hypothesis-driven slices (validation examples)."""
    out = defaultdict(set)
    for ex in examples:
        rid = ex["row_id"]
        out["all"].add(rid)
        ans_words = {t["word"] for a in ex["answers"] for t in D.normalise_tokens(D.tokenize(a))}
        if any(wvocab(w) == 1 for w in ans_words if normalize_answer(w)):
            out["A1_answer_oov"].add(rid)
        if _answer_is_handle_or_tag(ex):
            out["A2_handle_tag_username"].add(rid)
        if _answer_far_or_byline(ex):
            out["A3_far_or_byline"].add(rid)
        if not any(a.lower().strip() in ex["tweet"].lower() for a in ex["answers"]):
            out["A5_not_substring"].add(rid)
    return dict(out)


def slice_scores(pred_file, examples, slice_sets, metric="rougeL"):
    preds = {p["row_id"]: p["pred"] for p in json.loads(Path(pred_file).read_text())}
    per = {ex["row_id"]: score_example(preds.get(ex["row_id"], ""), ex["answers"], (metric,))[metric]
           for ex in examples}
    return {name: 100 * sum(per[r] for r in ids) / max(1, len(ids)) for name, ids in slice_sets.items()}


# ------------------------------------------------------------------ tables

def mean_sd(xs):
    return (statistics.mean(xs), statistics.stdev(xs) if len(xs) > 1 else 0.0)


def runs_by_name(names, split="val"):
    groups = defaultdict(list)
    for r in read_runs():
        if r["model"] in names and r["split"] == split:
            groups[r["model"]].append(r)
    return groups


def latex_table(rows, caption, label, cols=METRICS, header=None):
    """rows: [(label, {metric: (mean, sd) or value})]"""
    header = header or ["BLEU-1", "METEOR", "ROUGE-L", "EM", "F1"]
    lines = [r"\begin{table}[t]", r"\centering\small", r"\begin{tabular}{l" + "c" * len(cols) + "}",
             r"\toprule", "Model & " + " & ".join(header) + r" \\", r"\midrule"]
    for name, vals in rows:
        if name == "---":
            lines.append(r"\midrule")
            continue
        cells = []
        for c in cols:
            v = vals.get(c)
            if v is None:
                cells.append("--")
            elif isinstance(v, tuple):
                cells.append(f"{v[0]:.1f}$\\pm${v[1]:.1f}")
            else:
                cells.append(f"{v:.1f}")
        lines.append(name + " & " + " & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", rf"\caption{{{caption}}}", rf"\label{{{label}}}", r"\end{table}"]
    return "\n".join(lines)


# ------------------------------------------------------------------ figures

def _short(s, n=110):
    return s if len(s) <= n else s[:n - 1] + "…"


@torch.no_grad()
def attention_figure(model, ex, device, path, title_extra="", max_span_len=15):
    """Tweet x question similarity S and C2Q weights, plus Q2C and start/end probability strips."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    try:
        from .model import decode
    except ImportError:
        from model import decode
    model.eval()
    batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in D.collate([ex]).items()}
    o = model(batch, return_attn=True)
    (i, j), = decode(o["start_logits"], o["end_logits"], max_span_len)[0].tolist()
    pred = D.span_text(ex, i, j)
    t_lab = [t["raw"] for t in ex["t_tokens"]]
    q_lab = [t["raw"] for t in ex["q_tokens"]]
    c2q = o["c2q"][0].cpu().numpy()
    ps = torch.softmax(o["start_logits"][0], -1).cpu().numpy()
    pe = torch.softmax(o["end_logits"][0], -1).cpu().numpy()
    q2c = o["q2c"][0].cpu().numpy() if o["q2c"] is not None else None
    n_strips = 3 if q2c is not None else 2
    T = len(t_lab)
    fig = plt.figure(figsize=(max(8, 0.32 * T + 2), 3.2 + 0.28 * len(q_lab) + 0.35 * n_strips))
    gs = fig.add_gridspec(1 + n_strips, 1, height_ratios=[max(2, 0.28 * len(q_lab))] + [0.35] * n_strips, hspace=0.08)
    ax = fig.add_subplot(gs[0])
    ax.imshow(c2q.T, aspect="auto", cmap="Blues", vmin=0, vmax=1)
    ax.set_yticks(range(len(q_lab)), q_lab, fontsize=8)
    ax.set_xticks([])
    ax.set_ylabel("question (C2Q weights)", fontsize=8)
    strips = [("start", ps), ("end", pe)] + ([("Q2C", q2c)] if q2c is not None else [])
    for k, (name, v) in enumerate(strips):
        a = fig.add_subplot(gs[1 + k], sharex=ax)
        a.imshow(v[None, :T], aspect="auto", cmap="Oranges" if name != "Q2C" else "Greens", vmin=0)
        a.set_yticks([0], [name], fontsize=8)
        if k < len(strips) - 1:
            a.set_xticks([])
        else:
            a.set_xticks(range(T), t_lab, rotation=75, fontsize=7)
    for x in (i - 0.5, j + 0.5):
        ax.axvline(x, color="red", lw=1)
    gold = " | ".join(ex["answers"])
    ok = score_example(pred, ex["answers"], ("rougeL",))["rougeL"]
    fig.suptitle(f"Q: {_short(ex['question'])}\nTweet: {_short(ex['tweet'], 150)}\n"
                 f"gold: {gold}   pred: {pred}   (ROUGE-L {100*ok:.0f}) {title_extra}", fontsize=8, x=0.01, ha="left")
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    for ext in ("pdf", "png"):
        fig.savefig(f"{path}.{ext}", bbox_inches="tight", dpi=160)
    plt.close(fig)
    return pred


@torch.no_grad()
def self_attention_figure(model_on, model_off, ex, device, path):
    """Self-attention map of the full model, beside the start/end distributions with and without A3."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in D.collate([ex]).items()}
    model_on.eval(); model_off.eval()
    on = model_on(batch, return_attn=True)
    off = model_off(batch, return_attn=True)
    t_lab = [t["raw"] for t in ex["t_tokens"]]
    T = len(t_lab)
    sa = on["self_attn"][0].cpu().numpy()[:T, :T]
    fig, axes = plt.subplots(1, 2, figsize=(max(12, 0.5 * T), max(6, 0.3 * T)),
                             gridspec_kw=dict(width_ratios=[3, 1.2]))
    axes[0].imshow(sa, cmap="Purples")
    axes[0].set_xticks(range(T), t_lab, rotation=80, fontsize=7)
    axes[0].set_yticks(range(T), t_lab, fontsize=7)
    axes[0].set_title("gated self-attention (full model): row = query token", fontsize=9)
    for name, o, c in (("full: start", on, "C0"), ("full: end", on, "C1"), ("no self-attn: start", off, "C2"),
                       ("no self-attn: end", off, "C3")):
        key = "start_logits" if "start" in name else "end_logits"
        axes[1].plot(torch.softmax(o[key][0], -1).cpu().numpy()[:T], range(T), label=name, color=c,
                     ls="-" if "full" in name else "--")
    axes[1].invert_yaxis()
    axes[1].set_yticks(range(T), t_lab, fontsize=7)
    axes[1].legend(fontsize=7)
    axes[1].set_title("span probabilities", fontsize=9)
    fig.suptitle(f"Q: {ex['question']}   gold: {' | '.join(ex['answers'])}", fontsize=9)
    for ext in ("pdf", "png"):
        fig.savefig(f"{path}.{ext}", bbox_inches="tight", dpi=160)
    plt.close(fig)


# ------------------------------------------------------------------ errors

def export_error_sample(examples, pred_file, path, n=100, seed=4012):
    preds = {p["row_id"]: p["pred"] for p in json.loads(Path(pred_file).read_text())}
    errs = []
    for ex in examples:
        p = preds.get(ex["row_id"], "")
        s = score_example(p, ex["answers"], ("rougeL", "em"))
        if s["em"] < 1:
            errs.append((ex, p, s["rougeL"]))
    random.Random(seed).shuffle(errs)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["row_id", "question", "tweet", "gold", "pred", "rougeL", "extractive_gold",
                     "question_type", "error_cause"])
        for ex, p, r in errs[:n]:
            w.writerow([ex["row_id"], ex["question"], ex["tweet"], " | ".join(ex["answers"]), p, f"{100*r:.0f}",
                        int(ex["trainable"]), D._question_word(ex["question"]), ""])
    return len(errs)


Overwriting src/analysis.py


In [12]:
%%writefile scripts/stage5_tables.py
"""Stage 5: ablation table (mean ± sd over 3 seeds) and hypothesis-slice breakdowns from logged runs."""
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src import analysis as A  # noqa: E402
from src import data as D  # noqa: E402
from src.evaluate import METRICS  # noqa: E402

FROZEN = json.loads((ROOT / "results" / "FROZEN_CONFIG.json").read_text())
ABL = [("A5_single_span", "Full model v2"), ("S_A1_no_char", "A1: $-$ char-CNN"),
       ("S_A2_no_norm_flags", "A2: $-$ normaliser \\& flags"),
       ("S_A3_no_selfattn", "A3: $-$ self-attention"),
       ("S_A4_c2q_only", "A4: C2Q only (no Q2C)"),
       ("cand_lr5e-4_ft", "A5: wide soft targets (v1)"),
       ("B2_vanilla", "B2: all task-specific parts off")]

groups = A.runs_by_name([n for n, _ in ABL])
_, val_rows = D.get_splits()
prep = dict(use_normaliser=True, max_span_len=FROZEN["max_span_len"], tau=FROZEN["tau"])
va = D.preprocess(val_rows, prep)
tr_rows, _ = D.get_splits()
wv, _ = D.build_vocabs(D.preprocess(tr_rows, prep), min_freq=FROZEN.get("min_freq", 2))
sl = A.slices(va, wv)
print({k: len(v) for k, v in sl.items()})

rows, slice_rows, summary = [], [], {}
for name, label in ABL:
    rs = groups.get(name, [])
    if not rs:
        print("missing runs for", name)
        continue
    vals = {m: A.mean_sd([float(r[m]) for r in rs]) for m in METRICS}
    rows.append((f"{label} (n={len(rs)})", vals))
    per_slice = {}
    for r in rs:
        pf = ROOT / "results" / "preds" / f"{r['run_id']}.json"
        if pf.exists():
            for k, v in A.slice_scores(pf, va, sl).items():
                per_slice.setdefault(k, []).append(v)
    sv = {k: A.mean_sd(v) for k, v in per_slice.items()}
    slice_rows.append((label, sv))
    summary[name] = dict(metrics=vals, slices=sv, seeds=[r["seed"] for r in rs], run_ids=[r["run_id"] for r in rs])
    print(f"{name:18s} " + " ".join(f"{m}={vals[m][0]:.2f}±{vals[m][1]:.2f}" for m in METRICS))
    print(" " * 19 + " ".join(f"{k}={v[0]:.1f}" for k, v in sv.items()))

A.TAB_DIR.mkdir(parents=True, exist_ok=True)
(A.TAB_DIR / "ablations.tex").write_text(A.latex_table(
    rows, "Ablations on the validation split (mean $\\pm$ sd over 3 seeds; frozen configuration).", "tab:ablations"))
cols = ["all", "A1_answer_oov", "A2_handle_tag_username", "A3_far_or_byline", "A5_not_substring"]
hdr = [f"All ({len(sl['all'])})", f"OOV ans. ({len(sl['A1_answer_oov'])})",
       f"@/\\#/user ({len(sl['A2_handle_tag_username'])})", f"Far/byline ({len(sl['A3_far_or_byline'])})",
       f"Non-substr. ({len(sl['A5_not_substring'])})"]
(A.TAB_DIR / "ablation_slices.tex").write_text(A.latex_table(
    slice_rows, "Validation ROUGE-L on hypothesis slices (mean $\\pm$ sd over 3 seeds).", "tab:slices",
    cols=cols, header=hdr))
(ROOT / "results" / "stage5_summary.json").write_text(json.dumps(
    dict(summary=summary, slice_sizes={k: len(v) for k, v in sl.items()}), indent=1, default=str))
print("wrote tables/ablations.tex, tables/ablation_slices.tex")


Overwriting scripts/stage5_tables.py


In [13]:
%%writefile scripts/stage7_analysis.py
"""Stage 7: qualitative and error analysis on the validation split (frozen model, seed 1)."""
import collections
import csv
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src import analysis as A  # noqa: E402
from src import data as D  # noqa: E402
from src import train as T  # noqa: E402
from src.evaluate import evaluate, score_example  # noqa: E402
from src.runlog import read_runs  # noqa: E402

# A fresh Colab workspace does not contain these output directories yet.
# Create them before saving figures or the question-type LaTeX table.
A.FIG_DIR.mkdir(parents=True, exist_ok=True)
A.TAB_DIR.mkdir(parents=True, exist_ok=True)


def ckpt_for(model_name, seed=1):
    rs = [r for r in read_runs() if r["model"] == model_name and r["seed"] == str(seed) and r["split"] == "val"]
    return ROOT / "checkpoints" / f"{rs[-1]['run_id']}.pt", rs[-1]["run_id"]


device = T.get_device()
full_ck, full_id = ckpt_for("A5_single_span", 1)
noa3_ck, noa3_id = ckpt_for("S_A3_no_selfattn", 1)
model, cfg, wv, cv = T.load_checkpoint(full_ck, device)
model_off, cfg_off, wv_off, cv_off = T.load_checkpoint(noa3_ck, device)
assert wv.itos == wv_off.itos, "A3 model must share the vocabulary (same preprocessing and seed)"
_, val_rows = D.get_splits()
va = D.encode(D.preprocess(val_rows, dict(use_normaliser=cfg["use_normaliser"], max_span_len=cfg["max_span_len"],
                                          tau=cfg["tau"])), wv, cv)
preds, spans = T.predict(model, va, device, max_span_len=cfg["max_span_len"])
scores, per = evaluate(preds, [e["answers"] for e in va], return_per_example=True)
print("full model s1 val:", {k: round(v, 2) for k, v in scores.items()}, full_id)
pred_file = ROOT / "results" / "preds" / f"{full_id}.json"

# ---------------------------------------------------------------- pick examples (deterministic rules)
def short(e):
    return len(e["t_tokens"]) <= 32 and len(e["q_tokens"]) <= 12


def qtok_missing_but_similar(e):
    """A question content word that is not in the tweet but shares a 4-char prefix with a tweet token
    (a slang, misspelt or inflected keyword)."""
    tw = {t["word"] for t in e["t_tokens"]}
    for t in e["q_tokens"]:
        w = t["word"]
        if len(w) >= 4 and w not in D._STOP and w not in tw and any(x[:4] == w[:4] and x != w for x in tw):
            return True
    return False


rows = list(zip(va, preds, spans, per))
picked = {}
succ = [(e, p, s, sc) for e, p, s, sc in rows if sc["em"] == 1 and short(e)]
for qw in ("who", "what", "where"):
    c = [x for x in succ if A.D._question_word(x[0]["question"]) == qw]
    if c:
        picked[f"success_{qw}"] = c[0]
fail = [(e, p, s, sc) for e, p, s, sc in rows if sc["rougeL"] < 0.5 and short(e)]
rules = {
    "fail_username": lambda e: e["best"] is not None and (e["best"][0] >= e["byline_from"] or any(
        t["kind"] == "mention" for t in e["t_tokens"][e["best"][0]:e["best"][1] + 1])),
    "fail_abstractive": lambda e: not e["trainable"],
    "fail_slang_misspelt": qtok_missing_but_similar,
}
for name, rule in rules.items():
    c = [x for x in fail if rule(x[0]) and x[0]["row_id"] not in {v[0]["row_id"] for v in picked.values()}]
    if c:
        picked[name] = c[0]

for name, (e, p, s, sc) in picked.items():
    A.attention_figure(model, e, device, A.FIG_DIR / f"attn_{name}", max_span_len=cfg["max_span_len"])
    print(f"[{name}] Q: {e['question']} | gold: {e['answers']} | pred: {p} | rougeL={100*sc['rougeL']:.0f}")

# self-attention with vs without A3: an example whose answer is far from the question words or in the byline,
# and which the full model answers correctly
far = [x for x in rows if x[3]["em"] == 1 and A._answer_far_or_byline(x[0]) and short(x[0])]
if far:
    e = far[0][0]
    A.self_attention_figure(model, model_off, e, device, A.FIG_DIR / "self_attn_A3")
    print(f"[self_attn_A3] Q: {e['question']} | gold: {e['answers']}")

# ---------------------------------------------------------------- training curves for the frozen seeds
import matplotlib  # noqa: E402
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for r in [r for r in read_runs() if r["model"] == "A5_single_span"]:
    h = json.loads((ROOT / "results" / "history" / f"{r['run_id']}.json").read_text())["history"]
    ax[0].plot([x["epoch"] for x in h], [x["train_loss"] for x in h], label=f"seed {r['seed']}")
    ax[1].plot([x["epoch"] for x in h], [x["rougeL"] for x in h], label=f"seed {r['seed']}")
ax[0].set_title("training loss (single silver span)"); ax[1].set_title("validation ROUGE-L")
for a in ax:
    a.set_xlabel("epoch"); a.legend(fontsize=7)
for ext in ("pdf", "png"):
    fig.savefig(A.FIG_DIR / f"training_curves.{ext}", bbox_inches="tight", dpi=160)
plt.close(fig)

# ---------------------------------------------------------------- per-question-type table
by = collections.defaultdict(list)
for e, sc in zip(va, per):
    by[A.D._question_word(e["question"])].append(sc)
qt = {k: dict(n=len(v), **{m: 100 * sum(s[m] for s in v) / len(v) for m in ("rougeL", "f1", "em")})
      for k, v in sorted(by.items(), key=lambda kv: -len(kv[1]))}
for k, v in qt.items():
    print(f"qtype {k:6s} n={v['n']:4d} rougeL={v['rougeL']:.1f} f1={v['f1']:.1f} em={v['em']:.1f}")
lines = [r"\begin{table}[t]", r"\centering\small", r"\begin{tabular}{lrccc}", r"\toprule",
         r"Question word & $n$ & ROUGE-L & F1 & EM \\", r"\midrule"]
lines += [f"{k} & {v['n']} & {v['rougeL']:.1f} & {v['f1']:.1f} & {v['em']:.1f} \\\\" for k, v in qt.items()]
lines += [r"\bottomrule", r"\end{tabular}",
          r"\caption{Validation scores of the full model (seed 1) by question word.}", r"\label{tab:qtype}",
          r"\end{table}"]
(A.TAB_DIR / "qtype.tex").write_text("\n".join(lines))

# ---------------------------------------------------------------- error sample (+ automatic pre-tags)
n_err = A.export_error_sample(va, pred_file, ROOT / "results" / "error_sample.csv")
with open(ROOT / "results" / "error_sample.csv", encoding="utf-8") as f:
    err = list(csv.DictReader(f))
by_id = {e["row_id"]: e for e in va}
pred_by_id = {e["row_id"]: (p, s) for e, p, s in zip(va, preds, spans)}
auto = collections.Counter()
for r in err:
    e = by_id[int(r["row_id"])]
    p, (i, j) = pred_by_id[e["row_id"]]
    f1 = score_example(p, e["answers"], ("f1",))["f1"]
    if not e["trainable"]:
        tag = "abstractive gold (no good span)"
    elif 0 < f1:
        tag = "boundary / partial overlap"
    elif e["best"] and (e["best"][0] >= e["byline_from"]) != (i >= e["byline_from"]):
        tag = "byline vs body confusion"
    else:
        tag = "wrong span (no overlap)"
    r["auto_tag"] = tag
    auto[tag] += 1
with open(ROOT / "results" / "error_sample.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(err[0].keys()))
    w.writeheader()
    w.writerows(err)
print(f"errors (EM=0) on val: {n_err}/{len(va)}; automatic pre-tags of the 100-sample:", dict(auto))
(ROOT / "results" / "stage7_summary.json").write_text(json.dumps(dict(
    full_run=full_id, a3_run=noa3_id, val_scores=scores, picked={k: v[0]["row_id"] for k, v in picked.items()},
    qtype=qt, n_errors=n_err, auto_tags=auto), indent=1))


Overwriting scripts/stage7_analysis.py


In [14]:
Path('results/FROZEN_CONFIG.json').write_text('''{
 "_previous": "results/FROZEN_CONFIG_v1.json",
 "_rule": "one-factor sweep (seed 1); variants adopted only if val ROUGE-L > max of 3 base seeds; adopted lr=5e-4 and word_init=fasttext, confirmed with 3 seeds. v2 (2026-10-08, team decision): soft_spans=False, because the A5 ablation (single silver span) beat the soft-target model on all 3 seeds (val ROUGE-L 44.51\u00b10.11 vs 42.95\u00b10.47); ablations re-run on this base in GPU run 2",
 "_run_ids": [
  "A5_single_span-s1-20261007-141322",
  "A5_single_span-s3-20261007-141647",
  "A5_single_span-s2-20261007-141640"
 ],
 "_seeds": [
  1,
  3,
  2
 ],
 "_selected_from": "A5_single_span (Kaggle GPU run 1)",
 "_val_rougeL": [
  44.6186,
  44.5086,
  44.3927
 ],
 "attn_heads": 4,
 "batch_size": 32,
 "biattn_mode": "both",
 "char_dim": 32,
 "char_filters": 50,
 "char_widths": [
  3,
  4,
  5
 ],
 "dropout": 0.2,
 "epochs": 30,
 "grad_clip": 5.0,
 "hidden": 128,
 "lr": 0.0005,
 "max_span_len": 15,
 "min_freq": 2,
 "n_flags": 8,
 "patience": 4,
 "soft_spans": false,
 "tau": 0.5,
 "use_char_cnn": true,
 "use_normaliser": true,
 "use_self_attn": true,
 "use_token_flags": true,
 "weight_decay": 0.0,
 "word_dim": 100,
 "word_init": "fasttext"
}''')

1193

## Step 4 — A3 jobs and resume logic

In [15]:
# Frozen v2 configuration and experiment definitions.
FROZEN_V2 = json.loads(Path("results/FROZEN_CONFIG.json").read_text())
FROZEN_V2 = {k: v for k, v in FROZEN_V2.items() if not k.startswith("_")}
assert FROZEN_V2["soft_spans"] is False

MAIN_JOBS = [dict(name="A5_single_span", seed=s, set=FROZEN_V2) for s in (1, 2, 3)]
ABLATION_OVERRIDES = {
    "S_A1_no_char": {"use_char_cnn": False},
    "S_A2_no_norm_flags": {"use_normaliser": False, "use_token_flags": False},
    "S_A3_no_selfattn": {"use_self_attn": False},
    "S_A4_c2q_only": {"biattn_mode": "c2q"},
    "cand_lr5e-4_ft": {"soft_spans": True, "soft_tau": 0.5},  # A5: former v1
    "B2_vanilla": {"use_char_cnn": False, "use_normaliser": False,
                    "use_token_flags": False, "use_self_attn": False,
                    "biattn_mode": "c2q", "soft_spans": False},
}
unknown = set(ABLATIONS_TO_RUN) - set(ABLATION_OVERRIDES)
assert not unknown, f"Unknown ablations: {sorted(unknown)}"
ABLATION_JOBS = [
    dict(name=name, seed=s, set={**FROZEN_V2, **ABLATION_OVERRIDES[name]})
    for name in ABLATIONS_TO_RUN for s in (1, 2, 3)
]

def completed(name, seed):
    if not Path("results/runs.csv").exists():
        return False
    import csv
    return any(r["model"] == name and r["seed"] == str(seed) and r["split"] == "val"
               for r in csv.DictReader(open("results/runs.csv", encoding="utf-8")))

def run_job(job):
    if completed(job["name"], job["seed"]):
        print("SKIP completed:", job["name"], "seed", job["seed"])
        return
    args = [sys.executable, "-m", "src.train", "--name", job["name"],
            "--seed", str(job["seed"]), "--set"]
    args += [f"{k}={json.dumps(v)}" for k, v in job["set"].items()]
    print("RUN:", job["name"], "seed", job["seed"], flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    rc = subprocess.run(args, env=env).returncode
    assert rc == 0, f"Training failed: {job['name']} seed {job['seed']}"
    backup_artifacts()


## Step 5 — Confirm that the smoke test is disabled

In [16]:
if RUN_SMOKE_TEST:
    subprocess.run([sys.executable, "-m", "src.train", "--name", "smoke",
                    "--smoke", "--set", "lr=0.0005", "word_init=\"fasttext\"",
                    "soft_spans=false"], check=True)
    print("Smoke test passed")


## Step 6 — Confirm that v2 main-model retraining is disabled

In [17]:
if RUN_V2_MAIN:
    for job in MAIN_JOBS:
        run_job(job)
else:
    print("RUN_V2_MAIN=False: main-model training skipped")


RUN_V2_MAIN=False: main-model training skipped


## Step 7 — Train S_A3_no_selfattn — three seeds

In [18]:
if RUN_ABLATIONS:
    for job in ABLATION_JOBS:
        run_job(job)
else:
    print("RUN_ABLATIONS=False: optional ablations skipped")


SKIP completed: S_A3_no_selfattn seed 1
SKIP completed: S_A3_no_selfattn seed 2
SKIP completed: S_A3_no_selfattn seed 3


## Step 8 — Validation summary

In [19]:
import csv, statistics
rows = list(csv.DictReader(open("results/runs.csv", encoding="utf-8"))) if Path("results/runs.csv").exists() else []
groups = {}
for r in rows:
    if r["split"] == "val":
        groups.setdefault(r["model"], []).append(r)
for name, rs in groups.items():
    vals = [float(r["rougeL"]) for r in rs]
    tail = f" ± {statistics.stdev(vals):.2f}" if len(vals) > 1 else ""
    print(f"{name:24s} n={len(rs)} ROUGE-L={statistics.mean(vals):.2f}{tail}")
backup_artifacts()


A5_single_span           n=3 ROUGE-L=44.22 ± 0.16
S_A3_no_selfattn         n=3 ROUGE-L=42.41 ± 0.45
Backed up artifacts to /content/drive/MyDrive/CITS4012_A2_v2


## Step 9 — Generate attention figures and error analysis

In [20]:
# Run qualitative attention analysis only when the required checkpoints exist.
main_ready = completed("A5_single_span", 1)
a3_ready = completed("S_A3_no_selfattn", 1)
if main_ready and a3_ready:
    subprocess.run([sys.executable, "scripts/stage7_analysis.py"], check=True)
    backup_artifacts()
    print("Attention figures and error sample were generated.")
else:
    print("Stage 7 skipped. It requires A5_single_span seed 1 and S_A3_no_selfattn seed 1.")


Backed up artifacts to /content/drive/MyDrive/CITS4012_A2_v2
Attention figures and error sample were generated.


## Step 10 — Download updated artifacts

In [21]:
# Create a portable archive. Test evaluation is deliberately excluded.
import zipfile
archive = Path("/content/a2_v2_colab_artifacts.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in (WORK / "checkpoints", WORK / "results"):
        if folder.exists():
            for p in folder.rglob("*"):
                if p.is_file():
                    z.write(p, p.relative_to(WORK))
print(archive, round(archive.stat().st_size / 1e6, 1), "MB")
if BACKUP is not None:
    drive_archive = BACKUP / archive.name
    shutil.copy2(archive, drive_archive)
    print("Saved updated archive to Google Drive:", drive_archive)
from google.colab import files
files.download(str(archive))


/content/a2_v2_colab_artifacts.zip 88.2 MB
Saved updated archive to Google Drive: /content/drive/MyDrive/CITS4012_A2_v2/a2_v2_colab_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>